# Monarch Initiative — API & bulk

`biodb.monarch` exposes three query surfaces:

| Mode | Functions | What it answers |
|---|---|---|
| **BioLink REST** | `query_entity`, `query_associations`, `query_gene_associations` | "What is this CURIE?" / "what are its facts?" |
| **Cypher (Neo4j)** | `query_cypher`, `query_neighbors` | Graph-shaped questions — multi-hop, patterns, neighbourhoods |
| **Bulk TSV** | `get_gene_associations`, `read_causal_gene_to_disease_association` | Whole-corpus association tables for offline analysis |

The Monarch KG itself is reachable via Cypher — the per-association TSVs
are useful as a bulk download but are not the KG.

In [1]:
from biodb import monarch

## 1. BioLink v3 REST — one entity at a time

In [2]:
brca1 = monarch.query_entity("HGNC:1100")
{k: brca1.get(k) for k in ("id", "name", "category", "in_taxon_label")}

{'id': 'HGNC:1100',
 'name': 'BRCA1',
 'category': 'biolink:Gene',
 'in_taxon_label': 'Homo sapiens'}

In [3]:
page = monarch.query_associations(subject="HGNC:1100", limit=3)
print(f"total associations for BRCA1: {page['total']:,}")
for item in page["items"]:
    print(f"  {item['predicate']:35s} -> {item['object']:25s} {item.get('object_label', '')}")

total associations for BRCA1: 2,352
  biolink:interacts_with              -> HGNC:2197                 COL1A1
  biolink:interacts_with              -> HGNC:6827                 MAN2C1
  biolink:interacts_with              -> HGNC:6848                 MAP3K1


`query_gene_associations` paginates internally and returns a tidy
DataFrame (capped at ~1000 rows).

In [4]:
brca1_assoc = monarch.query_gene_associations("HGNC:1100", limit=100)
print(brca1_assoc.shape)
brca1_assoc[["subject", "predicate", "object", "object_label"]].head()

(1000, 92)


,subject,predicate,object,object_label
0,HGNC:1100,biolink:interacts_with,HGNC:2197,COL1A1
1,HGNC:1100,biolink:interacts_with,HGNC:6827,MAN2C1
2,HGNC:1100,biolink:interacts_with,HGNC:6848,MAP3K1
3,HGNC:1100,biolink:interacts_with,HGNC:7455,MT-ND1
4,HGNC:1100,biolink:interacts_with,HGNC:9311,PPP2R5C


## 2. Cypher against the public Monarch KG

Monarch runs a public read-only Neo4j at
`https://neo4j.monarchinitiative.org/` — no auth, no driver required.
`query_cypher` posts to the HTTP transactional endpoint and returns a
DataFrame.

In [5]:
monarch.query_cypher("MATCH (n) RETURN count(n) AS total")

,total
0,1460060


Use `$param` placeholders — never f-string interpolation (Cypher
injection is real).

In [6]:
monarch.query_cypher(
    "MATCH (g {id: $id})-[r]-(n) RETURN type(r) AS rel, n.category AS cat, "
    "count(*) AS n ORDER BY n DESC LIMIT 5",
    parameters={"id": "HGNC:1100"},
)

,rel,cat,n
0,biolink:interacts_with,"[biolink:Gene, biolink:GeneOrGeneProduct, biol...",2860
1,biolink:is_sequence_variant_of,"[biolink:SequenceVariant, biolink:GenomicEntit...",2336
2,biolink:enables,"[biolink:MolecularActivity, biolink:Occurrent,...",145
3,biolink:located_in,"[biolink:CellularComponent, biolink:Anatomical...",83
4,biolink:actively_involved_in,"[biolink:BiologicalProcess, biolink:Occurrent,...",62


`query_neighbors` is a convenience wrapper for the most common
graph-shaped question — "what is this CURIE connected to?".

In [7]:
monarch.query_neighbors("HGNC:1100", limit=5)

,predicate,neighbor_id,neighbor_name,neighbor_category
0,biolink:orthologous_to,Xenbase:XB-GENE-6254021,brca1.L,"[biolink:Gene, biolink:GeneOrGeneProduct, biol..."
1,biolink:orthologous_to,Xenbase:XB-GENE-490624,brca1,"[biolink:Gene, biolink:GeneOrGeneProduct, biol..."
2,biolink:orthologous_to,dictyBase:DDB_G0293300,DDB_G0293300,"[biolink:Gene, biolink:GeneOrGeneProduct, biol..."
3,biolink:orthologous_to,dictyBase:DDB_G0274195,DDB_G0274195,"[biolink:Gene, biolink:GeneOrGeneProduct, biol..."
4,biolink:orthologous_to,Xenbase:XB-GENE-490624,brca1,"[biolink:Gene, biolink:GeneOrGeneProduct, biol..."


## 3. Bulk association TSVs

The bulk download path is a thin reader over the Monarch ingest TSVs
at `https://data.monarchinitiative.org/`. Files are cached at
`~/.cache/monarch/`.

```python
from biodb import monarch

# Causal gene-to-disease association table (~few MB).
causal = monarch.read_causal_gene_to_disease_association()

# All gene-phenotype/disease associations (~tens of MB):
all_assoc = monarch.get_gene_associations(species="human")
```

These are *not* executed here — see the source-level docstrings for
species filters, dataset selection, and the polars/pandas knobs.